# Day 1: Setup, Python Refresher, and Your First Circuit

**Clemson Quantum Club** · SC Quantathon v3 Bootcamp · [clemsonquantum.com](https://clemsonquantum.com)

In this notebook:
1. Setup: install Qiskit, or open the notebook in qBraid Lab
2. Accounts: save your IBM Quantum credentials once
3. Python refresher, part 1: the parts of the language every later cell uses
4. Python refresher, part 2: numpy, complex numbers, and plots
5. Bits and gates: what a classical computer does, written as tables
6. Qubits: what changes, in words
7. Your first circuit: a one-qubit quantum random number generator on a simulator
8. The same circuit on a real quantum computer (optional)

> Cells marked **Your turn** have a few lines for you to fill in. Cells marked **Checkpoint** contain `assert` statements: if the cell runs without an error, the answer above it is correct. This is the solutions notebook: every Your turn cell is filled in and the hardware run is switched on.

## 1. Setup

Two ways to run this notebook. Every cell works on both.

**Local** (a conda environment, from the repository root):
```bash
conda create -n scqv3 python=3.12
conda activate scqv3
pip install -r requirements.txt
jupyter lab
```

**qBraid Lab** (no install): sign in at [account.qbraid.com](https://account.qbraid.com), choose a compute profile, click **Launch**, and open this notebook in the default Python environment. It should have Qiskit, Aer, and the IBM connector; the next cell prints what it finds.

The next cell detects which path you are on and prints versions.

In [ ]:
import os
from platform import python_version

import qiskit
import qiskit_aer
import qiskit_ibm_runtime

ON_QBRAID = os.path.exists("/opt/.qbraid") or any("QBRAID" in k for k in os.environ)
print(f"Path:               {'qBraid Lab' if ON_QBRAID else 'local'}")
print(f"Python              {python_version()}")
print(f"qiskit              {qiskit.__version__}")
print(f"qiskit-aer          {qiskit_aer.__version__}")
print(f"qiskit-ibm-runtime  {qiskit_ibm_runtime.__version__}")

assert tuple(int(x) for x in qiskit.__version__.split(".")[:2]) >= (2, 0), "Need Qiskit 2.x"

# Hardware calls are opt-in. Leave False to use a cached result in section 8 (nothing can stall).
# Flip to True once your IBM account is saved and you want to spend about 2 seconds of QPU time.
RUN_ON_HARDWARE = False

SHOTS = 1000

## 2. Accounts

### 2.1 IBM Quantum

1. Sign in at [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com).
2. From the dashboard, create an **API key** and copy it.
3. Open the **Instances** page from the main menu, create an instance on the **Open Plan**, and copy its **CRN**. The CRN is optional: without it, the service picks your Open Plan instance itself.
4. Run the cell below. It asks for the key and the CRN in hidden prompts (press Enter at the CRN prompt to skip it), saves them on disk under the name `scqv3`, and every later notebook loads them from there.

> ⚠️ Treat the API key and CRN like passwords. Do not share this notebook without removing your API key and CRN. If you prefer a file, put `IBM_API_KEY=...` and `IBM_CRN=...` in a `.env` at the repository root; that file is listed in `.gitignore`.

In [ ]:
import os
from getpass import getpass

from qiskit_ibm_runtime import QiskitRuntimeService

def read_env_file(path=".env"):
    values = {}
    if os.path.exists(path):
        for line in open(path):
            if "=" in line and not line.startswith("#"):
                key, _, value = line.strip().partition("=")
                values[key] = value.strip().strip('"')
    return values

try:
    QiskitRuntimeService(name="scqv3")
    print("Account 'scqv3' is already saved. Nothing to do.")
except Exception:
    env = {**read_env_file(), **os.environ}
    try:
        api_key = env.get("IBM_API_KEY") or getpass("IBM Quantum API key (hidden): ")
        crn     = env.get("IBM_CRN")     or getpass("Instance CRN (hidden, Enter to skip): ")
    except Exception:
        api_key = crn = ""           # no prompt available (for example, a headless run)
    if api_key:
        QiskitRuntimeService.save_account(
            channel="ibm_quantum_platform",
            token=api_key,
            instance=crn or None,
            name="scqv3",
            overwrite=True,
        )
        print("Account saved as 'scqv3'.")
    else:
        print("Skipped: no key entered. Run this cell again when you have one.")

In [ ]:
# Checkpoint: load the saved account and list the machines you can use.
# If no account is saved yet, this prints a note and moves on; section 8 then uses a cached result.
try:
    service = QiskitRuntimeService(name="scqv3")
    backends = service.backends(operational=True, simulator=False)
    assert len(backends) > 0, "No hardware backends visible: check the instance CRN"
    for b in backends:
        print(f"{b.name:<16} {b.num_qubits:>4} qubits   queue: {b.status().pending_jobs}")
except Exception as e:
    service = None
    print("No saved IBM account found:", type(e).__name__)

### 2.2 qBraid

Create a free account at [account.qbraid.com](https://account.qbraid.com) with the email you registered with. Accepted participants are added to the SC Quantathon v3 organization there, which is where the hackathon credits live. Day 3 walks through the platform.

## 3. Python refresher, part 1: the language

Every later cell in this bootcamp is built from the pieces in this section. Run each cell and read the output before moving on.

### 3.1 Variables and types

A variable is a name for a value. Python figures out the type from the value.

In [ ]:
n_qubits = 2          # int
prob = 0.5            # float
label = "00"          # str
is_zero = True        # bool

print(n_qubits, type(n_qubits))
print(prob, type(prob))
print(label, type(label))
print(is_zero, type(is_zero))
print(2 ** n_qubits, "states for", n_qubits, "qubits")   # ** is exponent

### 3.2 Lists

A list is an ordered collection. Indexing starts at 0, negative indices count from the end, and `a:b` slices from `a` up to but not including `b`.

In [ ]:
outcomes = ["00", "01", "10", "11"]

print(outcomes[0])        # first
print(outcomes[-1])       # last
print(outcomes[1:3])      # second and third
print(len(outcomes))      # how many

outcomes.append("100")    # lists can grow
print(outcomes)

### 3.3 Loops and conditions

`for` repeats a block once per item. `range(n)` counts from 0 to n - 1. `if` / `else` chooses between blocks. Indentation is what marks the block.

In [ ]:
for i in range(4):
    if i % 2 == 0:            # % is remainder
        print(i, "is even")
    else:
        print(i, "is odd")

total = 0
for x in [3, 1, 4, 1, 5]:
    total = total + x
print("sum:", total)

### 3.4 Functions

`def` names a block of code that takes inputs and returns an output. Writing a function once and calling it many times is how the exercises in later days are structured.

In [ ]:
def probability_of_zero(counts, shots):
    return counts.get("0", 0) / shots

def normalize_counts(counts):
    shots = sum(counts.values())
    return {key: value / shots for key, value in counts.items()}

example = {"0": 489, "1": 511}
print(probability_of_zero(example, 1000))
print(normalize_counts(example))

### 3.5 Dictionaries

A dictionary maps keys to values. Every measurement result in Qiskit is a dictionary from an outcome string to the number of times it appeared, so you will read these constantly.

In [ ]:
counts = {"00": 503, "11": 497}

print(counts["00"])              # value for a key
print(counts.get("01", 0))       # 0 if the key is missing, instead of an error
print(list(counts.keys()))
print(list(counts.values()))
print(sum(counts.values()))      # total shots

for outcome, n in counts.items():
    print(outcome, "appeared", n, "times")

counts["01"] = 3                 # add a key
print(sorted(counts.items(), key=lambda kv: kv[1], reverse=True))   # sort by value, largest first

### 3.6 f-strings

An f-string puts values inside text. After a colon you can set the format: `.3f` means three decimals, `.1%` means percent with one decimal, `>8` right-aligns in eight characters.

In [ ]:
p = 0.48713
name = "ibm_torino"
qubits = 133

print(f"P(0) = {p}")
print(f"P(0) = {p:.3f}")
print(f"P(0) = {p:.1%}")
print(f"{name:<12} {qubits:>5} qubits")

### 3.7 Imports

Code that lives in another package is brought in with `import`. `import numpy as np` gives the package a short name; `from qiskit import QuantumCircuit` brings in one object.

In [ ]:
import math
import numpy as np
from qiskit import QuantumCircuit

print(math.sqrt(2))
print(np.sqrt(2))
print(QuantumCircuit)

### Your turn: the most common outcome

Write `most_common(counts)` so that it returns the key with the largest value.

In [ ]:
def most_common(counts):
    best_key = None    
    
    ### WRITE YOUR CODE BELOW HERE ###

    ### YOUR CODE FINISHES HERE ###
    return best_key

In [ ]:
# Checkpoint
assert most_common({"00": 503, "11": 497}) == "00", "most_common is not finished: fill in the cell above"
assert most_common({"0": 12, "1": 988}) == "1"
print("most_common works")

## 4. Python refresher, part 2: numpy, complex numbers, and plots

### 4.1 Arrays

A numpy array is a list that does arithmetic element by element. Qubit states and gates are arrays.

In [ ]:
a = np.array([1, 2, 3])
b = np.array([10, 20, 30])

print(a + b)          # elementwise
print(2 * a)
print(a * b)          # elementwise product, not the dot product
print(a @ b)          # dot product: 1*10 + 2*20 + 3*30
print(a.shape, a.dtype)

### 4.2 Complex numbers

Python writes the imaginary unit as `1j`. Three facts about a complex number $z = x + iy$ are used every day from here on: its magnitude $|z| = \sqrt{x^2 + y^2}$, its conjugate $\bar z = x - iy$, and $|z|^2 = z \bar z$. A complex number of magnitude 1 can be written $e^{i\phi}$ for some angle $\phi$.

In [ ]:
z = 3 + 4j

print("z         =", z)
print("|z|       =", abs(z))
print("conj(z)   =", np.conj(z))
print("z*conj(z) =", z * np.conj(z), "   (real, equal to |z|^2)")
print("angle     =", np.angle(z), "radians")

phi = 0.7
u = np.exp(1j * phi)          # e^{i phi}
print("|e^{i phi}| =", abs(u))

### 4.3 Vectors and matrices

A column vector is a 1D array. A matrix is a 2D array. `@` multiplies them. `.conj().T` is the conjugate transpose, written $A^\dagger$. Compare floats with `np.allclose`, never with `==`.

The example matrix rotates the plane by a quarter turn: it sends $(1, 0)$ to $(0, 1)$, and undoing it is the same as applying its transpose.

In [ ]:
e1 = np.array([1, 0])
e2 = np.array([0, 1])
R = np.array([[0, -1],
              [1,  0]])

print("R e1      =", R @ e1)
print("R e2      =", R @ e2)
print("R^T R     =\n", R.T @ R)
print("is R^T R the identity?", np.allclose(R.T @ R, np.eye(2)))

v = np.array([1 + 1j, 2 - 1j])
print("v^dagger  =", v.conj().T)
print("|v|^2     =", np.sum(np.abs(v) ** 2))

### 4.4 A bar chart

Results in this bootcamp are histograms. Qiskit has a helper for the common case, but plain matplotlib is enough to make one.

In [ ]:
import matplotlib.pyplot as plt

counts = {"00": 503, "01": 12, "10": 9, "11": 476}

plt.bar(counts.keys(), counts.values())
plt.xlabel("outcome")
plt.ylabel("counts")
plt.title("A histogram, by hand")
plt.show()

### Your turn: normalize a vector

A vector is normalized when the sum of $|v_i|^2$ over its entries is 1. Divide `v` by its length so that `v_norm` is normalized.

In [ ]:
v = np.array([3, 4j])
v_norm = None

### WRITE YOUR CODE BELOW HERE ###

### YOUR CODE FINISHES HERE ###

In [ ]:
# Checkpoint
assert v_norm is not None, "v_norm is not set: fill in the cell above"
assert np.isclose(np.sum(np.abs(v_norm) ** 2), 1), "sum of |v_i|^2 should be 1"
print("v_norm =", v_norm, "  sum of |v_i|^2 =", np.sum(np.abs(v_norm) ** 2))

## 5. Bits and gates

### 5.1 A bit is a coin lying flat

A bit is always exactly one of two things, 0 or 1. Read it as often as you like; the answer never changes. In hardware it is a transistor, a switch that either passes current or blocks it.

### 5.2 Gates are lookup tables

A classical logic gate is a table: bits in, bits out. NOT flips a bit. AND outputs 1 only when both inputs are 1. Chain enough of them and you get addition, then a CPU.

In [ ]:
NOT = {0: 1, 1: 0}
AND = {(0, 0): 0, (0, 1): 0, (1, 0): 0, (1, 1): 1}

print("NOT:", NOT)
print("AND:", AND)
print("AND(1, 0) =", AND[(1, 0)])

From `AND = 0` you cannot recover the inputs: three different input pairs give that output. Most classical gates throw information away, so they are **irreversible**. NOT is the exception: apply it twice and you are back where you started.

Quantum gates are also tables, written as matrices, and every one of them is reversible.

### Your turn: XOR

XOR outputs 1 when the inputs differ. Fill in its table, then check the two facts below: XOR alone is irreversible, but XOR together with one of its inputs is reversible.

In [ ]:
XOR = {
    ### WRITE YOUR CODE BELOW HERE ###

    ### YOUR CODE FINISHES HERE ###
}

In [ ]:
# Checkpoint: XOR alone loses information ...
assert len(XOR) == 4, "XOR is not finished: fill in the cell above"
assert XOR[(0, 1)] == XOR[(1, 0)] == 1 and XOR[(0, 0)] == XOR[(1, 1)] == 0

# ... but (a, a XOR b) keeps it: all four outputs are different, so the inputs can be recovered
outputs = {(a, XOR[(a, b)]) for (a, b) in XOR}
assert len(outputs) == 4
print("XOR with one input kept is reversible:", sorted(outputs))

## 6. Qubits

### 6.1 A qubit is a coin still spinning

Until you look, a qubit is a weighted blend of 0 and 1, called a **superposition**. Looking, which is called **measurement**, forces a single answer and destroys the blend. Run the same circuit a thousand times and you get a histogram, not a number. That is the normal way to read a quantum computer.

Where the analogy breaks: a qubit also carries a **phase**, a sign attached to each possibility, and signs can cancel. Coins do not do that.

### 6.2 Reading a machine through a histogram

A fair classical coin, flipped a thousand times, already produces the kind of output you will see from a quantum computer: counts that hover near 50/50 without landing on it.

In [ ]:
rng = np.random.default_rng(seed=7)          # seeded, so this cell always prints the same thing

flips = rng.integers(0, 2, size=1000)          # 1000 draws of 0 or 1
coin_counts = {"0": int(np.sum(flips == 0)), "1": int(np.sum(flips == 1))}
print(coin_counts)

plt.bar(coin_counts.keys(), coin_counts.values())
plt.title("A classical coin, 1000 flips")
plt.show()

### 6.3 Three things a quantum computer has that yours does not

- **Superposition**: one qubit holds a weighted blend of 0 and 1; $n$ qubits hold a blend of all $2^n$ bitstrings at once.
- **Entanglement**: qubits whose outcomes are linked, so measuring one tells you the other.
- **Interference**: the phases can be arranged so that wrong answers cancel and the right one adds up.

A quantum algorithm is a recipe that uses all three and then measures. Today's circuit uses only the first.

## 7. Your first circuit: a quantum random number generator

### 7.1 Build it

The smallest useful quantum program: one qubit, one gate, one measurement. The **Hadamard** gate $H$ turns $|0\rangle$ into an equal blend of $|0\rangle$ and $|1\rangle$, and the measurement picks one.

$$|0\rangle \xrightarrow{\;H\;} \tfrac{1}{\sqrt2}\big(|0\rangle + |1\rangle\big) \xrightarrow{\text{measure}} 0 \text{ or } 1$$

In Qiskit, a circuit is an object you add operations to, one line each. `QuantumCircuit(1, 1)` means one qubit and one classical bit to store the measured value.

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(1, 1)     # 1 qubit, 1 classical bit
qc.h(0)                       # Hadamard on qubit 0
qc.measure(0, 0)              # measure qubit 0 into classical bit 0

qc.draw("mpl")

Read the drawing left to right: the qubit `q` starts in $|0\rangle$, passes through $H$, and is measured into the classical bit `c`.

### 7.2 Run it

A **simulator** computes what the circuit does on a classical computer. `AerSimulator` is the one shipped with Qiskit. Running a circuit `shots` times returns the counts dictionary from section 3.5.

In [ ]:
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

simulator = AerSimulator()
result = simulator.run(qc, shots=SHOTS).result()
counts = result.get_counts()

print(counts)
plot_histogram(counts, title=f"One-qubit QRNG, {SHOTS} shots")

Those three lines are the whole pattern for every circuit in this bootcamp: build a circuit, run it on a backend, read the counts.

### 7.3 Shots

One run gives one bit. The probabilities only show up over many runs. With $N$ shots and probability $p$ per outcome, the count fluctuates by about one standard deviation,

$$\sigma = \sqrt{N p (1 - p)},$$

which for $N = 1000$ and $p = 1/2$ is about 16. A result of 484 or 516 zeros is ordinary; 400 would not be.

In [ ]:
# Checkpoint: the counts are within a few sigma of 500/500
assert set(counts) <= {"0", "1"}
assert 440 < counts.get("0", 0) < 560, f"Suspiciously biased: {counts}"

sigma = np.sqrt(SHOTS * 0.5 * 0.5)
print(f"sigma = {sigma:.1f} counts")

for shots in [10, 100, 1000, 10000]:
    c = simulator.run(qc, shots=shots).result().get_counts()
    print(f"{shots:>6} shots: P(0) = {c.get('0', 0) / shots:.3f}")

The same experiment over many values of $N$, plotted against the one-standard-deviation line $\sqrt{p(1-p)/N}$:

In [ ]:
shots_list = np.unique(np.logspace(1, 5, 25).astype(int))
errors = []
for n in shots_list:
    c = simulator.run(qc, shots=int(n)).result().get_counts()
    errors.append(abs(c.get("0", 0) / n - 0.5))

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(shots_list, errors, "o", ms=4, label="|P(0) estimate - 1/2|")
ax.plot(shots_list, 0.5 / np.sqrt(shots_list), "-", color="0.4", label="one standard deviation")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("shots N"); ax.set_ylabel("error in P(0)")
ax.legend(frameon=False)
plt.show()

The estimate of $P(0)$ tightens as shots grow, at the rate $\sigma / N = \sqrt{p(1-p)/N}$. Ten times more shots buys about three times more precision.

### 7.4 Where the randomness comes from

Python's `random` module is a **pseudorandom** generator: give it the same seed and it repeats the same sequence forever. The Aer simulator is also a classical program, so it also has a seed. Only on real hardware is the outcome undetermined until the measurement happens.

In [ ]:
import random

random.seed(1); a = [random.randint(0, 1) for _ in range(8)]
random.seed(1); b = [random.randint(0, 1) for _ in range(8)]
print("random.seed(1) twice:      ", a, b, "identical:", a == b)

seeded = AerSimulator(seed_simulator=1)
c1 = seeded.run(qc, shots=8, memory=True).result().get_memory()
seeded = AerSimulator(seed_simulator=1)
c2 = seeded.run(qc, shots=8, memory=True).result().get_memory()
print("AerSimulator seed=1 twice: ", c1, c2, "identical:", c1 == c2)

unseeded = AerSimulator()
print("AerSimulator, no seed:     ", unseeded.run(qc, shots=8, memory=True).result().get_memory())

`memory=True` keeps the outcome of every individual shot, in order, instead of only the totals. That is the tool for the next exercise.

### 7.5 Your turn: a 32-bit random integer

Run the circuit for 32 shots with `memory=True`, join the outcomes into one string of 32 characters, and convert it to an integer with `int(bits, 2)`.

In [ ]:
bits = ""
number = None

### WRITE YOUR CODE BELOW HERE ###

### YOUR CODE FINISHES HERE ###

In [ ]:
# Checkpoint
assert len(bits) == 32, "bits should have 32 characters: fill in the cell above"
assert set(bits) <= {"0", "1"}
assert 0 <= number < 2 ** 32
print(bits, "->", number)

### 7.6 Your turn: remove the Hadamard

Build the same circuit without the `h` gate and run it for 1000 shots. Without the gate the qubit never leaves $|0\rangle$, so every shot should read 0.

In [ ]:
counts_no_h = None

### WRITE YOUR CODE BELOW HERE ###

### YOUR CODE FINISHES HERE ###

In [ ]:
# Checkpoint
assert counts_no_h is not None, "counts_no_h is not set: fill in the cell above"
assert counts_no_h == {"0": 1000}, counts_no_h
print(counts_no_h)

## 8. The same circuit on a real quantum computer (optional)

The simulator is the math. A real machine is a chip in a refrigerator, and the same circuit has to be rewritten into the gates that chip physically has before it can run. Two things change in the code: the circuit goes through a **pass manager** (the transpiler) for the chosen backend, and it is submitted through a **Sampler** instead of `simulator.run`.

This section costs about 2 seconds of the 10 free minutes of QPU time per 28-day rolling window. It runs only if `RUN_ON_HARDWARE = True` in section 1 and an account was found in section 2; otherwise it uses a result cached from an earlier run so the comparison below always works.

In [ ]:
from qiskit import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler

job = None
if RUN_ON_HARDWARE and service is not None:
    backend = service.least_busy(operational=True, simulator=False, min_num_qubits=1)
    print(f"Using {backend.name} ({backend.num_qubits} qubits, {backend.status().pending_jobs} jobs queued)")

    pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
    qc_isa = pm.run(qc)
    print("Gates the chip will run:", dict(qc_isa.count_ops()))

    job = Sampler(mode=backend).run([qc_isa], shots=SHOTS)
    print("Job id:", job.job_id(), " status:", job.status())
else:
    print("RUN_ON_HARDWARE is False or no account saved: using the cached hardware result below.")

On a superconducting chip the Hadamard is not a native gate, so `count_ops()` shows it rewritten as `rz` and `sx` rotations. Day 5 covers why.

The cell below returns your job's counts if the job has finished, and the cached counts otherwise. Re-run it later to swap in your own data.

In [ ]:
# Recorded run: ibm_kingston (a Heron r2 processor), physical qubit 0, 1000 shots
CACHED_QPU_COUNTS = {"0": 295, "1": 705}

def qpu_counts_or_cached(job):
    if job is None:
        return CACHED_QPU_COUNTS
    try:
        if job.status() == "DONE":
            print(f"Using your result from {job.backend().name}")
            return job.result()[0].data.c.get_counts()      # 'c' is the classical register of qc
        print(f"Job is {job.status()}; using the cached result for now. Re-run this cell later.")
    except Exception as e:
        print("Could not fetch the job:", e)
    return CACHED_QPU_COUNTS

qpu_counts = qpu_counts_or_cached(job)
print(qpu_counts)

In [ ]:
plot_histogram(
    [coin_counts, counts, qpu_counts],
    legend=["classical coin", "AerSimulator", "real QPU"],
    title=f"One random bit, {SHOTS} draws each: classical, simulator, hardware",
)

## 9. Summary

- A bit is one of two values; a classical gate is a lookup table, and most of them are irreversible.
- A qubit is a superposition of 0 and 1 until it is measured. Measurement gives one bit; repeating gives a histogram.
- A quantum circuit in Qiskit is built one operation per line, run on a backend with a number of shots, and read as a counts dictionary.
- Counts fluctuate by about $\sqrt{Np(1-p)}$; the estimate of a probability improves as $1/\sqrt{N}$.
- Simulators are classical programs with seeds. Randomness that is not reproducible comes only from measuring real hardware.
- Running on hardware adds a transpile step and a Sampler, and the result carries readout error.

## Further reading

- [Hello World](https://quantum.cloud.ibm.com/docs/en/tutorials/hello-world), the IBM tutorial this notebook follows
- [Basics of Quantum Information](https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information), John Watrous, for the math of Day 2 onward
- [Python for Beginners](https://docs.python.org/3/tutorial/) and the [NumPy quickstart](https://numpy.org/doc/stable/user/quickstart.html)
- [Qiskit documentation](https://quantum.cloud.ibm.com/docs)